In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
# parameter setting
batch_size = 32
learning_rate = 0.001
epoch = 10

# GPU setting
cuda_available = torch.cuda.is_available()
device = torch.device('cuda' if cuda_available else 'cpu')

print('Current cuda device is', device)


Current cuda device is cuda


In [3]:
# dataset 불러오기
train_data = datasets.MNIST(root = './data/train/',
                            train=True,
                            transform=transforms.ToTensor(),
                            download=True)
test_data = datasets.MNIST(root = './data/test/',
                           train=False,
                           transform=transforms.ToTensor(),
                           download=True)

train_loader=DataLoader(dataset=train_data,
                        batch_size=batch_size,
                        shuffle=True)

test_loader=DataLoader(dataset=test_data,
                       batch_size=batch_size,
                       shuffle=True)


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 482kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.5MB/s]
100%|██████████| 9.91M/9.91M [00:00<00:00, 16.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 481kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.48MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.88MB/s]


In [4]:
#Deep Learning Model setting
class Convolution_Neural_Networks(nn.Module):
  def __init__(self):
    super(Convolution_Neural_Networks, self).__init__()
    # [conv1] 입력 -> (1, 28, 28) -> (32, 28, 28)
    self.conv1=nn.Conv2d(in_channels = 1, out_channels = 32, kernel_size = 3, stride = 1, padding='same')
    # [conv2] 입력 -> (32, 28, 28) -> (64, 28, 28)
    self.conv2=nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3, stride = 1, padding='same')
    # dropout: 학습 중 일부 채널을 0으로 만들어 과적합을 줄이려는 기법
    self.dropout=nn.Dropout2d(0.25) # eval()에서 적용 X
    self.relu=nn.ReLU()
    self.max_pooling=nn.MaxPool2d(kernel_size=2)
    self.fc1=nn.Linear(3136,1000) # 7*7*64 = 3136
    # 최종 정답 레이어는 10가지
    self.fc2=nn.Linear(1000,10)

  def forward(self, x):
    x=self.conv1(x) # (1, 28, 28) -> (32, 28, 28)
    x=self.relu(x) # (32, 28, 28)
    x=self.max_pooling(x) # (32, 28, 28) -> (32, 14, 14)

    x=self.conv2(x) # (32, 14, 14) -> (64, 14, 14)
    x=self.relu(x) # (64, 14, 14)
    x=self.max_pooling(x) #(64, 14, 14) -> (64, 7, 7)

    x=self.dropout(x)
    x=torch.flatten(x,1) # (64, 7, 7) -> 1차원(3136)

    x=self.fc1(x) # 3136 -> 1000
    x=self.relu(x) # 1000
    x=self.fc2(x) # 1000 -> 10
    output = F.log_softmax(x, dim=1)
    return output


In [5]:
# model에 GPU 적용
model=Convolution_Neural_Networks().cuda(device)
# mdoel에 CPU 적용
#model = Convolution_Neural_Networks().cpu()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [6]:
# Train
i = 1
for epoch in range(epoch):
  for data, target in train_loader: 
      # GPU 학습을 위해 텐서를 GPU로 이동
      # (batch_size, 1, 28, 28)
      data=data.to(device)
      # (batch_size, )
      target=target.to(device)

      # 기울기를 0으로 초기화
      optimizer.zero_grad()
      output = model(data) # forward
      loss=criterion(output, target) # 손실함수 계산
      loss.backward() # 역전파
      optimizer.step() # 가중치 업데이트
      if i%1000 == 0:
        print("Train Step : {}\tLoss : {:3f}".format(i, loss.item()))
      i += 1

Train Step : 1000	Loss : 0.002819
Train Step : 2000	Loss : 0.086429
Train Step : 3000	Loss : 0.088629
Train Step : 4000	Loss : 0.003567
Train Step : 5000	Loss : 0.129005
Train Step : 6000	Loss : 0.001656
Train Step : 7000	Loss : 0.009128
Train Step : 8000	Loss : 0.016869
Train Step : 9000	Loss : 0.001860
Train Step : 10000	Loss : 0.002124
Train Step : 11000	Loss : 0.008134
Train Step : 12000	Loss : 0.000383
Train Step : 13000	Loss : 0.000682
Train Step : 14000	Loss : 0.007408
Train Step : 15000	Loss : 0.000512
Train Step : 16000	Loss : 0.001222
Train Step : 17000	Loss : 0.007785
Train Step : 18000	Loss : 0.000071


In [7]:
# Test
model.eval() # test 모드
correct = 0
for data, target in test_loader:
  data=data.to(device)
  target=target.to(device)
  output=model(data)
  prediction=output.data.max(1)[1]
  # eq: pytorch의 배열 요소 별로 비교하는 메서드.
  correct += prediction.eq(target.data).sum()
print('Test set Accuracy: {:.2f}%'.format(100.*correct / len(test_loader.dataset)))

Test set Accuracy: 99.16%


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
#from google.colab import drive
#drive.flush_and_unmount()

In [11]:
# Model Save
PATH = '/content/drive/MyDrive/CNN for num classification_20240418a.pt'
torch.save(model.state_dict(), PATH)

In [12]:
!nvidia-smi

Wed May 13 12:25:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             28W /   70W |     251MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----